In [0]:
# Databricks notebook source
# ============================================================
# 1. CARGA DE TABLAS Y CONFIGURACIÓN DE MLFLOW
# ============================================================
from pyspark.sql import functions as F
import mlflow
import mlflow.spark

from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

TRAIN_TABLE = "workspace.bigdata_daigreco.tp_final_train"
TEST_TABLE = "workspace.bigdata_daigreco.tp_final_test"

train_df = spark.table(TRAIN_TABLE)
test_df = spark.table(TEST_TABLE)

print(f"Train cargado: {train_df.count():,} registros")
print(f"Test cargado:  {test_df.count():,} registros")

experiment_name = "/Shared/tp_final_bank_marketing_mlflow"
mlflow.set_experiment(experiment_name)
print(f"Experimento MLflow activo: {experiment_name}")

# COMMAND ----------

# ============================================================
# 2. DEFINICIÓN DE EVALUADORES Y MÉTRICAS
# ============================================================
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="DEPOSIT",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="DEPOSIT",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="DEPOSIT",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="DEPOSIT",
    predictionCol="prediction",
    metricName="f1"
)

def positive_class_metrics(predictions):
    tp = predictions.filter((F.col("DEPOSIT") == 1) & (F.col("prediction") == 1)).count()
    fp = predictions.filter((F.col("DEPOSIT") == 0) & (F.col("prediction") == 1)).count()
    fn = predictions.filter((F.col("DEPOSIT") == 1) & (F.col("prediction") == 0)).count()
    tn = predictions.filter((F.col("DEPOSIT") == 0) & (F.col("prediction") == 0)).count()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_pos = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision_positive": precision,
        "recall_positive": recall,
        "f1_positive": f1_pos
    }

# COMMAND ----------

# ============================================================
# 3. MODELO 1: LOGISTIC REGRESSION (BASELINE)
# ============================================================
lr = LogisticRegression(
    featuresCol="features",
    labelCol="DEPOSIT",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

with mlflow.start_run(run_name="logistic_regression_baseline"):
    lr_model = lr.fit(train_df)
    preds_lr = lr_model.transform(test_df)

    roc_auc = roc_evaluator.evaluate(preds_lr)
    pr_auc = pr_evaluator.evaluate(preds_lr)
    acc = accuracy_evaluator.evaluate(preds_lr)
    f1_w = f1_evaluator.evaluate(preds_lr)
    pos_m = positive_class_metrics(preds_lr)

    # Log Parámetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("maxIter", 100)
    mlflow.log_param("regParam", 0.0)

    # Log Métricas
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1_w)
    mlflow.log_metric("precision_positive", pos_m["precision_positive"])
    mlflow.log_metric("recall_positive", pos_m["recall_positive"])
    mlflow.log_metric("f1_positive", pos_m["f1_positive"])

    print("--- LOGISTIC REGRESSION ---")
    print(f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | Accuracy: {acc:.4f} | F1+: {pos_m['f1_positive']:.4f}")

# COMMAND ----------

# ============================================================
# 4. MODELO 2: RANDOM FOREST (BASELINE)
# ============================================================
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="DEPOSIT",
    numTrees=100,
    maxDepth=5,
    seed=42
)

with mlflow.start_run(run_name="random_forest_baseline"):
    rf_model = rf.fit(train_df)
    preds_rf = rf_model.transform(test_df)

    roc_auc = roc_evaluator.evaluate(preds_rf)
    pr_auc = pr_evaluator.evaluate(preds_rf)
    acc = accuracy_evaluator.evaluate(preds_rf)
    f1_w = f1_evaluator.evaluate(preds_rf)
    pos_m = positive_class_metrics(preds_rf)

    # Log Parámetros
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 5)

    # Log Métricas
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1_w)
    mlflow.log_metric("precision_positive", pos_m["precision_positive"])
    mlflow.log_metric("recall_positive", pos_m["recall_positive"])
    mlflow.log_metric("f1_positive", pos_m["f1_positive"])

    print("--- RANDOM FOREST ---")
    print(f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | Accuracy: {acc:.4f} | F1+: {pos_m['f1_positive']:.4f}")

# COMMAND ----------

# ============================================================
# 5. MODELO 3: GRADIENT-BOOSTED TREES (BASELINE)
# ============================================================
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="DEPOSIT",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    seed=42
)

with mlflow.start_run(run_name="gradient_boosted_trees_baseline"):
    gbt_model = gbt.fit(train_df)
    preds_gbt = gbt_model.transform(test_df)

    roc_auc = roc_evaluator.evaluate(preds_gbt)
    pr_auc = pr_evaluator.evaluate(preds_gbt)
    acc = accuracy_evaluator.evaluate(preds_gbt)
    f1_w = f1_evaluator.evaluate(preds_gbt)
    pos_m = positive_class_metrics(preds_gbt)

    # Log Parámetros
    mlflow.log_param("model", "GBTClassifier")
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("stepSize", 0.1)

    # Log Métricas
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1_w)
    mlflow.log_metric("precision_positive", pos_m["precision_positive"])
    mlflow.log_metric("recall_positive", pos_m["recall_positive"])
    mlflow.log_metric("f1_positive", pos_m["f1_positive"])

    print("--- GRADIENT-BOOSTED TREES ---")
    print(f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | Accuracy: {acc:.4f} | F1+: {pos_m['f1_positive']:.4f}")

print("\n✅ Todos los modelos baseline fueron entrenados y logueados en MLflow.")

{"ts": "2026-10-30 20:48:43.625", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3MzcyODk0NjI5MzUyMRABIAEyJDAxYTE4ZTkyLTI2M2ItNzEyZS04YjFmLTE4NTczMTllMzBmODokNjhmYjFmNzMtNTU0Ni0zNTQxLTg5MDMtY2M2Nzk4OGJlOWYxSgsImYWU1wYQwJWMRlABWAFgAWiNyuDDhcWjDYgBAA==.", "context": {}}
{"ts": "2026-10-30 20:48:43.625", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3MzcyODk0NjI5MzUyMRABIAEyJDAxYTE4ZTkyLTI2M2ItNzEyZS04YjFmLTE4NTczMTllMzBmODokNjhmYjFmNzMtNTU0Ni0zNTQxLTg5MDMtY2M2Nzk4OGJlOWYxSgsImYWU1wYQwJWMRlABWAFgAWiNyuDDhcWjDYgBAA==.", "context": {}}
{"ts": "2026-10-30 20:48:43.625", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3MzcyODk0NjI5MzUyMRABIAEyJDAxYTE4ZTkyLTI2M2ItNzEyZS04YjFmLTE4NTczMTllMzBmODokNjhmYjFmNzMtNTU0Ni0zNTQxLTg5MDMtY2M2Nzk4

Train cargado: 32,903 registros
Test cargado:  8,285 registros


2026/10/30 20:48:54 INFO mlflow.tracking.fluent: Experiment with name '/Shared/tp_final_bank_marketing_mlflow' does not exist. Creating a new experiment.


Experimento MLflow activo: /Shared/tp_final_bank_marketing_mlflow


2026/10/30 20:48:55 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
2026/10/30 20:49:29 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


--- LOGISTIC REGRESSION ---
ROC-AUC: 0.9312 | PR-AUC: 0.5712 | Accuracy: 0.9079 | F1+: 0.5003


2026/10/30 20:49:47 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


--- RANDOM FOREST ---
ROC-AUC: 0.9177 | PR-AUC: 0.5675 | Accuracy: 0.9004 | F1+: 0.2614
--- GRADIENT-BOOSTED TREES ---
ROC-AUC: 0.9410 | PR-AUC: 0.6250 | Accuracy: 0.9130 | F1+: 0.5654

✅ Todos los modelos baseline fueron entrenados y logueados en MLflow.
